<a href="https://colab.research.google.com/github/sigmue01/presentations/blob/main/IVIM_CRLB_paper_ranges_fixedvalues.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

b-value optimization code

Author :  Diego Hernando
Editors :  Oscar Janelfjord, Eric E. Sigmund

These scripts were composed adopting a Cramer-Rao Lower Bound (CRLB) formalism to provide guidance on selecting optimum b-values for IVIM MRI experiments in a organ-adaptive manner. The scripts also allow for fixing some b-values and optimizing the remaining members of given set.



Mean values and ranges for 6 different organ systems (brain, kidney, Liver, Muscle, Breast, Pancreas) were drawn from recent reviews or meta-analyses below.


(Brain)

Vieni, C., et al., Effect of intravoxel incoherent motion on diffusion parameters in normal brain. Neuroimage, 2020. 204: p. 116228.

(Kidney)

Ljimani, A., et al., Consensus-based technical recommendations for clinical translation of renal diffusion-weighted MRI. MAGMA, 2020. 33(1): p. 177-195.

(Liver)

Li, Y.T., et al., Liver intravoxel incoherent motion (IVIM) magnetic resonance imaging: a comprehensive review of published data on normal values and applications for fibrosis and tumor evaluation. Quant Imaging Med Surg, 2017. 7(1): p. 59-78.

(Muscle)

Englund, E.K., et al., Intravoxel Incoherent Motion Magnetic Resonance Imaging in Skeletal Muscle: Review and Future Directions. J Magn Reson Imaging, 2022. 55(4): p. 988-1012.

(Breast)

Liang, J., et al., Intravoxel Incoherent Motion Diffusion-Weighted Imaging for Quantitative Differentiation of Breast Tumors: A Meta-Analysis. Front Oncol, 2020. 10: p. 585486.

(Pancreas)

Zhu, M., et al., Accuracy of quantitative diffusion-weighted imaging for differentiating benign and malignant pancreatic lesions: a systematic review and meta-analysis. European Radiology, 2021. 31(10): p. 7746-7759.




In [ ]:
import numpy as np
from scipy.stats import rice
from scipy.special import i0
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib


def ivim_signal(b, S0, D, f, D_star):
    """
    Calculates the IVIM signal under the bi-exponential model.

    Args:
        b (array-like): b-values
        S0 (float): Initial signal amplitude
        D (float): True diffusion coefficient
        f (float): Perfusion fraction
        D_star (float): Pseudodiffusion coefficient

    Returns:
        array-like: IVIM signal values
    """
    return S0 * (f * np.exp(-b * D_star) + (1 - f) * np.exp(-b * D))

def jacobian(b, S0, D, f, D_star):
    """
    Calculates the Jacobian matrix of the IVIM signal model.

    Args:
        b (array-like): b-values
        S0 (float): Initial signal amplitude
        D (float): True diffusion coefficient
        f (float): Perfusion fraction
        D_star (float): Pseudodiffusion coefficient

    Returns:
        numpy.ndarray: Jacobian matrix
    """
    dSdS0 = f * np.exp(-b * D_star) + (1 - f) * np.exp(-b * D)
    dSdD = -S0 * (1 - f) * b * np.exp(-b * D)
    dSdF = S0 * (np.exp(-b * D_star) - np.exp(-b * D))
    dSdDstar = -S0 * f * b * np.exp(-b * D_star)
    return np.stack([dSdS0, dSdD, dSdF, dSdDstar], axis=1)


def crlb(b, S0, D, f, D_star, sigma):
    """
    Calculates the Cramer-Rao Lower Bound (CRLB) for IVIM parameters.
    With the assumption of Gaussian noise (so this approximation
    does not account for noise floor effects)
    Without accounting for TE-lengthening required for higher b values

    Args:
        b (array-like): b-values
        S0 (float): Initial signal amplitude
        D (float): True diffusion coefficient
        f (float): Perfusion fraction
        D_star (float): Pseudodiffusion coefficient
        sigma (float): Standard deviation of Rician noise

    Returns:
        numpy.ndarray: Covariance matrix (the diagonal elements are CRLBs)
    """
    J = jacobian(b, S0, D, f, D_star)
    fisher_info = (J.T @ J) / sigma**2
    return np.linalg.inv(fisher_info)

# Example usage:
b_values = np.array([0, 10, 100, 200, 500, 800])
true_S0 = 1.0  # mm^2/s
true_D = 0.001  # mm^2/s
true_f = 0.15
true_D_star = 0.02  # mm^2/s
sigma = 0.05  # Standard deviation of noise

crlb_matrix = crlb(b_values, true_S0, true_D, true_f, true_D_star, sigma)

std_D = np.sqrt(crlb_matrix[1,1]);
std_f = np.sqrt(crlb_matrix[2,2]);
std_D_star = np.sqrt(crlb_matrix[3,3]);

#print("Sqrt CRLB for D:", std_D)
#print("Sqrt CRLB for f:", std_f)
#print("Sqrt CRLB for D*:", std_D_star)


error_metric = std_D/true_D + std_f +  std_D_star/true_D_star # Not normalizing by true_f, which can be zero
print("Error metric", error_metric)


Error metric 2.514208947502162


In [ ]:
# Cost function that penalizes standard deviations predicted by the CRLB
def objective_function_range(b, S0s, Ds, fs, D_stars, sigma):
    b = np.clip(b,0,1500)
    error_metric = 0.0
    error_metrics = []
    for S0 in S0s:
        for D in Ds:
            for f in fs:
                for D_star in D_stars:
                    crlb_matrix = crlb(b, S0, D, f, D_star, sigma)
                    std_D = np.sqrt(crlb_matrix[1,1]);
                    std_f = np.sqrt(crlb_matrix[2,2]);
                    std_D_star = np.sqrt(crlb_matrix[3,3]);
                    cur_error =  std_D/D + std_f +  std_D_star/D_star # Not normalizing by true_f, which can be zero
                    error_metrics = np.append(error_metrics,(cur_error))
    # Maximum error across all true parameter settings (could also take the average, or some other combination)
    error_metric = np.max(error_metrics)
    return error_metric

# Cost function that penalizes standard deviations predicted by the CRLB, with some fixed b values
def objective_function_range_fixed_tuneable_b(b1,b2, S0s, Ds, fs, D_stars, sigma):
    b = np.concatenate([b1,b2])
    error_metric = objective_function_range(b, S0s, Ds, fs, D_stars, sigma)
    return error_metric


# Measures of uncertainty for each of the IVIM parameters
def uncertainty_measures(b, S0s, Ds, fs, D_stars, sigma):
    b = np.clip(b,0,1500)
    error_metric = 0.0
    error_metrics = []
    error_metrics_D = []
    error_metrics_f = []
    error_metrics_D_star = []
    for S0 in S0s:
        for D in Ds:
            for f in fs:
                for D_star in D_stars:
                    crlb_matrix = crlb(b, S0, D, f, D_star, sigma)
                    std_D = np.sqrt(crlb_matrix[1,1]);
                    std_f = np.sqrt(crlb_matrix[2,2]);
                    std_D_star = np.sqrt(crlb_matrix[3,3]);
                    cur_error =  std_D/D + std_f +  std_D_star/D_star # Not normalizing by true_f, which can be zero
                    error_metrics_D = np.append(error_metrics_D,(std_D/D))
                    error_metrics_D_star = np.append(error_metrics_D_star,(std_D_star/D_star))
                    error_metrics_f = np.append(error_metrics_f,(std_f))
                    error_metrics = np.append(error_metrics,(cur_error))
    ## Maximum error across all true parameter settings (could also take the average, or some other combination)
    ##error_metric = np.max(error_metrics)
    # Average error across all true parameter settings
    error_metric = np.mean(error_metrics)
    return error_metric, error_metrics_D, error_metrics_f, error_metrics_D_star


In [ ]:
# True parameters from meta-analysis cited in IVIM white paper (July 2024)
trueParams= np.dtype([('name', (np.str_, 30)), ('D', np.float64),('f', np.float64), ('Dstar', np.float64),
                      ('SD_D', np.float64),('SD_f', np.float64), ('SD_Dstar', np.float64)])
myTrueParams = np.array([('Kidney Cortex 1.5T', 1.966/1000, 19.9/100, 50.8/1000, 0.075/1000, 3.2/100, 13.45/1000),
                         ('Kidney Cortex 3.0T', 1.919/1000, 20.1/100, 24.964/1000, 0.229/1000, 8.4/100,20.3/1000 ),
                         ('Kidney Medulla 1.5T', 1.884/1000, 17.5/100, 57.35/1000, 0.076/1000, 5.5/100, 25.5/1000),
                         ('Kidney Medulla 3.0T', 1.796/1000, 18/100, 29.02/1000, 0.228/1000, 7.8/100, 19.27/1000),
                         ('Kidney Average', 1.891/1000, 18.875/100, 40.534/1000, 0.152/1000, 6.225/100, 19.63/1000),
                         ('Liver', 1.09/1000, 23.05/100, 70.02/1000,0.17/1000, 8.48/100, 31.01/1000),
                         ('Muscle', 1.47/1000, 10.34/100, 30.88/1000, 0.28/1000, 5.92/100, 39.18/1000 ),
                         ('Breast Malignant', 0.9676/1000, 11.305/100, 37.76/1000, 0.32/1000, 4.506/100, 19.12/1000),
                         ('Breast Benign', 1.4305/1000, 7.004/100, 52.33/1000, 0.37/1000, 4.22/100, 28.50/1000),
                          ('Breast average', 1.199/1000, 9.15/100, 45.05/1000,2*0.34/1000, 2*4.37/100, 2*23.81/1000), # doubled average SD to include B and M values
                         ('Pancreas Malignant', 1.396/1000, 12.394/100, 22.16/1000,0.48/1000, 4.97/100, 12.83/1000),
                         ('Pancreas Benign', 1.409/1000, 20.033/100, 25.39/1000,0.72/1000, 7.99/100, 13.69/1000),
                          ('Pancreas average', 1.402/1000, 16.21/100, 23.77/1000,2*0.60/1000, 2*6.48/100, 2*13.26/1000), # doubled average SD to include B and M values
                          ('Brain', 0.833/1000, 7.64/100, 10.88/1000,0.074/1000, 3.15/100, 4.78/1000),
                        ],
       dtype=trueParams)


In [ ]:
# For each example of tissue parameters, optimize the b values
FIX_SOME_B = 1
b2 = np.array([0,200,800]); # Fixed b values (these should eventually be different for different organs)
num_b = 6
num_b1 = num_b - b2.size # Number of b values to actually determine (if some are fixed)
allb = np.zeros([myTrueParams.size,num_b])
for k in range(myTrueParams.size):

    # Collect current tissue parameters
    print("Current tissue parameters:", myTrueParams[k])
    true_S0s = np.array([1.0])
    true_Ds = np.array([myTrueParams[k]['D']])
    true_fs = np.array([myTrueParams[k]['f']])
    true_D_stars = np.array([myTrueParams[k]['Dstar']])
    sigma = 0.04  # Standard deviation of noise (assuming SNR = true_S0s / sigma for the b=0 signal)

    if FIX_SOME_B > 0:
        # Define fixed parameters for optimization
        fixed_params = {'b2': b2,'S0s': true_S0s, 'Ds': true_Ds, 'fs': true_fs, 'D_stars':true_D_stars, 'sigma':sigma}

        # Choose an optimization method and provide an initial guess for x
        method = 'Nelder-Mead'  # Other options: 'differential_evolution', etc.
        b0 = np.logspace(0, 2.5, num_b1, base=10.0)

        # Minimize the objective function (using args to pass fixed parameters)
        result = minimize(objective_function_range_fixed_tuneable_b, b0, method=method, args=(fixed_params['b2'],fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(result.x,0,2500)
        b0 = b_opt
        result = minimize(objective_function_range_fixed_tuneable_b, b0, method=method, args=(fixed_params['b2'],fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(np.concatenate([result.x,b2]),0,2500)
        b_opt = np.sort(b_opt)

    else:
        # Define fixed parameters for optimization
        fixed_params = {'S0s': true_S0s, 'Ds': true_Ds, 'fs': true_fs, 'D_stars':true_D_stars, 'sigma':sigma}

        # Choose an optimization method and provide an initial guess for x
        method = 'Nelder-Mead'  # Other options: 'differential_evolution', etc.
        b0 = np.logspace(0, 2.5, num_b, base=10.0)

        # Minimize the objective function (using args to pass fixed parameters)
        result = minimize(objective_function_range, b0, method=method, args=(fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(result.x,0,2500)
        b0 = b_opt
        result = minimize(objective_function_range, b0, method=method, args=(fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(result.x,0,2500)
        b_opt = np.sort(b_opt)

    # Check if the optimization was successful
    if not result.success:
        print("Optimization failed!")

    print("Optimized b values:", np.round(b_opt))
    allb[k,:] = b_opt

    # Calculate some measure of uncertainty for each of the parameters in each of the organs
    # Note that these measures of uncertainty scale with the noise assumed in the signal, so the absolute values need to be interpreted with care
    error_metric, error_metrics_D, error_metrics_f, error_metrics_D_star = uncertainty_measures(b_opt, true_S0s, true_Ds, true_fs, true_D_stars, sigma)
    print("Uncertainty for D (stdD/D):",  "{:.2f}".format(np.mean(error_metrics_D)))
    print("Uncertainty for f (stdf):",  "{:.2f}".format(np.mean(error_metrics_f)))
    print("Uncertainty for Dstar (stdDstar/Dstar):",  "{:.2f}".format(np.mean(error_metrics_D_star)))



Current tissue parameters: ('Kidney Cortex 1.5T', 0.001966, 0.199, 0.0508, 7.5e-05, 0.032, 0.01345)
Optimized b values: [  0.   0.  17. 100. 200. 800.]
Uncertainty for D (stdD/D): 0.18
Uncertainty for f (stdf): 0.07
Uncertainty for Dstar (stdDstar/Dstar): 0.73
Current tissue parameters: ('Kidney Cortex 3.0T', 0.001919, 0.201, 0.024964, 0.000229, 0.084, 0.0203)
Optimized b values: [  0.   0.  32. 172. 200. 800.]
Uncertainty for D (stdD/D): 0.21
Uncertainty for f (stdf): 0.09
Uncertainty for Dstar (stdDstar/Dstar): 0.81
Current tissue parameters: ('Kidney Medulla 1.5T', 0.001884, 0.175, 0.05735, 7.6e-05, 0.055, 0.0255)
Optimized b values: [  0.   0.  15.  93. 200. 800.]
Uncertainty for D (stdD/D): 0.17
Uncertainty for f (stdf): 0.06
Uncertainty for Dstar (stdDstar/Dstar): 0.82
Current tissue parameters: ('Kidney Medulla 3.0T', 0.001796, 0.18, 0.02902, 0.000228, 0.078, 0.01927)
Optimized b values: [  0.   0.  28. 158. 200. 800.]
Uncertainty for D (stdD/D): 0.19
Uncertainty for f (stdf): 0

In [ ]:
allb = np.round(allb*10)/10
for k in range(myTrueParams.size):
    print(allb[k,0],  allb[k,1],allb[k,2],allb[k,3] )


0.0 0.0 16.5 100.1
0.0 0.0 32.2 171.8
0.0 0.0 14.8 93.4
0.0 0.0 28.4 157.5
0.0 0.0 20.6 120.2
0.0 13.2 13.2 82.7
0.0 0.0 27.5 163.6
0.0 0.0 23.4 149.9
0.0 0.0 16.7 111.5
0.0 0.0 37.1 200.0
0.0 0.0 32.9 184.7


In [ ]:
# For each example of tissue parameters, optimize the b values
# In this case, we will include a range of D, f, and D* values in the optimization
# based on the SDs provided in meta-analyses cited in IVIM white paper
FIX_SOME_B = 1
b2 = np.array([0,200,800]); # Fixed b values (these should eventually be different for different organs)
num_b = 6
num_b1 = num_b - b2.size # Number of b values to actually determine (if some are fixed)
allb = np.zeros([myTrueParams.size,num_b])
for k in range(myTrueParams.size):

    # Collect current tissue parameters
    print("Current tissue parameters:", myTrueParams[k])

    # Set the ranges of true tissue parameters to include, based on the mean and SD for each
    # Basically, the ranges are (mean-SD, mean+SD) for each parameter, with some sanity checks
    curD = myTrueParams[k]['D']
    curSD_D = myTrueParams[k]['SD_D']
    curf = myTrueParams[k]['f']
    curSD_f = myTrueParams[k]['SD_f']
    curDstar = myTrueParams[k]['Dstar']
    curSD_Dstar = myTrueParams[k]['SD_Dstar']
    minD = max(0.0,curD-curSD_D)
    maxD = curD+curSD_D
    minf = max(0.0,curf-curSD_f)
    maxf = min(1.0,curf+curSD_f)
    minDstar = max(10/1000,curDstar-curSD_Dstar) # Let us avoid very small Dstar values, which complicate IVIM fitting
    maxDstar = curDstar+curSD_Dstar

    # Define some arrays with the true tissue parameters for the CRLB optimization
    true_S0s = np.array([1.0])
    true_Ds = np.linspace(minD, maxD, num=5, endpoint=True)
    true_fs = np.linspace(minf, maxf, num=3, endpoint=True)
    true_D_stars = np.linspace(minDstar, maxDstar, num=11, endpoint=True)
    sigma = 0.04  # Standard deviation of noise (assuming SNR = true_S0s / sigma for the b=0 signal)


    if FIX_SOME_B > 0:
        # Define fixed parameters for optimization
        fixed_params = {'b2': b2,'S0s': true_S0s, 'Ds': true_Ds, 'fs': true_fs, 'D_stars':true_D_stars, 'sigma':sigma}

        # Choose an optimization method and provide an initial guess for x
        method = 'Nelder-Mead'  # Other options: 'differential_evolution', etc.
        b0 = np.logspace(0, 2.5, num_b1, base=10.0)
        b0all = np.concatenate([b0,b2])
        b0all = np.sort(b0all)
        b0all=[0,30,70,100,200,800]

        # Minimize the objective function (using args to pass fixed parameters)
        result = minimize(objective_function_range_fixed_tuneable_b, b0, method=method, args=(fixed_params['b2'],fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(result.x,0,800)
        b0 = b_opt
        result = minimize(objective_function_range_fixed_tuneable_b, b0, method=method, args=(fixed_params['b2'],fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        result = minimize(objective_function_range_fixed_tuneable_b, result.x, method=method, args=(fixed_params['b2'],fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        result = minimize(objective_function_range_fixed_tuneable_b, result.x, method=method, args=(fixed_params['b2'],fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(np.concatenate([result.x,b2]),0,800)
        b_opt = np.sort(b_opt)

    else:

        # Define fixed parameters for optimization
        fixed_params = {'S0s': true_S0s, 'Ds': true_Ds, 'fs': true_fs, 'D_stars':true_D_stars, 'sigma':sigma}

        # Choose an optimization method and provide an initial guess for x
        method = 'Nelder-Mead'  # Other options: 'differential_evolution', etc.
        b0 = np.logspace(0, 2.5, num_b, base=10.0)
        b0all = b0

        # Minimize the objective function (using args to pass fixed parameters)
        result = minimize(objective_function_range, b0, method=method, args=(fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(result.x,0,800)
        b0 = b_opt
        result = minimize(objective_function_range, b0, method=method, args=(fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        result = minimize(objective_function_range, result.x, method=method, args=(fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        result = minimize(objective_function_range, result.x, method=method, args=(fixed_params['S0s'], fixed_params['Ds'], fixed_params['fs'], fixed_params['D_stars'], fixed_params['sigma']),options={'maxiter': 50000, 'maxfev': 200000})
        b_opt = np.clip(result.x,0,800)
        b_opt = np.sort(b_opt)

    # Check if the optimization was successful
    if not result.success:
        print("Optimization failed!")

    print("Initial b values:", np.round(b0all))
    print("Optimized b values:", np.round(b_opt))
    allb[k,:] = b_opt

    # Calculate some measure of uncertainty for each of the parameters in each of the organs
    # Note that these measures of uncertainty scale with the noise assumed in the signal, so the absolute values need to be interpreted with care
    error_metric0, error_metrics_D0, error_metrics_f0, error_metrics_D_star0 = uncertainty_measures(b0all, true_S0s, true_Ds, true_fs, true_D_stars, sigma)
    error_metric, error_metrics_D, error_metrics_f, error_metrics_D_star = uncertainty_measures(b_opt, true_S0s, true_Ds, true_fs, true_D_stars, sigma)
    print("Uncertainty for D (stdD/D):",  "{:.2f}".format(np.mean(error_metrics_D)), "  Initial uncertainty: "   ,  "{:.2f}".format(np.mean(error_metrics_D0)) )
    print("Uncertainty for f (stdf):",  "{:.2f}".format(np.mean(error_metrics_f)), "  Initial uncertainty: "   ,  "{:.2f}".format(np.mean(error_metrics_f0)))
    print("Uncertainty for Dstar (stdDstar/Dstar):",  "{:.2f}".format(np.mean(error_metrics_D_star)), "  Initial uncertainty: "   ,  "{:.2f}".format(np.mean(error_metrics_D_star0)))



Current tissue parameters: ('Kidney Cortex 1.5T', 0.001966, 0.199, 0.0508, 7.5e-05, 0.032, 0.01345)
Initial b values: [  0  30  70 100 200 800]
Optimized b values: [  0.   0.  14. 200. 800. 800.]
Uncertainty for D (stdD/D): 0.16   Initial uncertainty:  0.17
Uncertainty for f (stdf): 0.09   Initial uncertainty:  0.06
Uncertainty for Dstar (stdDstar/Dstar): 0.88   Initial uncertainty:  0.91
Current tissue parameters: ('Kidney Cortex 3.0T', 0.001919, 0.201, 0.024964, 0.000229, 0.084, 0.0203)
Initial b values: [  0  30  70 100 200 800]
Optimized b values: [  0.   0.  51. 200. 317. 800.]
Uncertainty for D (stdD/D): 0.23   Initial uncertainty:  0.22
Uncertainty for f (stdf): 0.13   Initial uncertainty:  0.12
Uncertainty for Dstar (stdDstar/Dstar): 1.42   Initial uncertainty:  1.13
Current tissue parameters: ('Kidney Medulla 1.5T', 0.001884, 0.175, 0.05735, 7.6e-05, 0.055, 0.0255)
Initial b values: [  0  30  70 100 200 800]
Optimized b values: [  0.  13.  26. 113. 200. 800.]
Uncertainty for D